# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, following the structure recommended in the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values (unique identifiers in Croissant).

In [ ]:
# List all available record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Record Sets found ({len(record_sets)}):")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', None)}")
    print(f"  description: {getattr(rs, 'description', None)}")
    print(f"  Fields/columns:")
    for field in rs.fields:
        print(f"    - @id: {field.id}")
        print(f"      name: {getattr(field, 'name', None)}")
        print(f"      dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare dataframes for all available record sets
dataframes = {}

record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set: {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# For demonstration, select the first loaded record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. Reference fields by their `@id`. Edit variable assignments as needed based on data overview above.

In [ ]:
# Choose a numeric field for filtering, normalization, and grouping.
# Replace the following example field IDs with the correct ones for your dataset.
numeric_field_id = None  # Set to e.g. '@id' of a numeric field such as 'coverage', 'peptide_count', etc.
group_field_id = None    # Set to group by another field, such as an experimental condition or protein class

if dataframes:
    df = dataframes[main_record_set_id]
    # Suggest a numeric candidate automatically if not assigned
    if numeric_field_id is None:
        # Try to auto-detect float columns (basic heuristic)
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    print(f"Using numeric field: {numeric_field_id}")

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
        if threshold is not None:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
            display(filtered_df.head())

            # Normalization
            normalized_field = f"{numeric_field_id}_normalized"
            filtered_df[normalized_field] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, normalized_field]].head())

            # Group by a field (choose a categorical field if present)
            if group_field_id is None:
                # Try to auto-detect an object-type column with a reasonable number of groups
                for col in df.columns:
                    if col != numeric_field_id and df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < 16:
                        group_field_id = col
                        break
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
                print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
                display(grouped_df.head())
            else:
                print('No suitable group field found for grouping operation.')
        else:
            print(f"Numeric field {numeric_field_id} is not numeric. Skipping EDA.")
    else:
        print('No numeric fields found in the dataframe.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we provide an example histogram and boxplot, using the same field `@id` as before.

In [ ]:
# Visualize distributions if a numeric field is available
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
This notebook demonstrated how to load and explore the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` library.

Key steps:
- Loaded dataset via its Croissant schema URL.
- Inspected record sets and field `@id`s.
- Loaded records into DataFrames and performed simple EDA (filtering, normalization, grouping).
- Produced simple visualizations for selected numeric fields.

For further analysis, consult the Croissant schema for field definitions and consider extending the EDA and visualization sections as needed.